# Session 2 — NumPy Fundamentals
**Data Analysis in Python** · Economic Analytics II (MSc) · Gdańsk Tech

*Onsite, Thu 8 Oct 2026 (semester week 2), 08:15–10:50, room 812 B · live-coding notebook*

---

**Learning objectives.** By the end of this session you will be able to:
1. Explain why NumPy arrays, not lists, are the basic container for numerical data, and predict how they behave under arithmetic (*Understand*).
2. Create and inspect arrays (`shape`, `dtype`) and select from them with slices and boolean masks (*Apply*).
3. Compute economic transformations of whole series in one line — growth rates, log differences, per-capita values, shares — without loops (*Apply*).
4. Use broadcasting to combine arrays of different shapes, and diagnose the error when shapes do not fit (*Apply / Analyse*).

**We start with the Session 1 self-check quiz on eNauczanie (5 minutes), then come back here.**

Yesterday you computed growth rates with a `for` loop over a list. Today the same calculations become one line each — and run fast on a million numbers.


## 0. Setup and data

Same dataset as Session 1: 203 quarters of the US economy, 1959Q1–2009Q3. We use pandas only to read the file and immediately turn the columns into NumPy arrays — pandas proper starts in Session 3.


In [13]:
import os
DATA = "us_macro_1959_2009.csv"
if not os.path.exists(DATA):
    try:
        from google.colab import files          # only exists inside Colab
        print("Running in Colab - choose", DATA, "from your computer:")
        files.upload()
    except ImportError:
        print(f"{DATA} not found - put it in the same folder as this notebook, then re-run this cell.")
print("Data file found:", os.path.exists(DATA))



Data file found: True


In [14]:
import numpy as np
import pandas as pd

macro = pd.read_csv("us_macro_1959_2009.csv")

year  = macro["year"].to_numpy()
qtr   = macro["quarter"].to_numpy()
gdp   = macro["realgdp"].to_numpy()     # billions of chained 2005 USD
cons  = macro["realcons"].to_numpy()
inv   = macro["realinv"].to_numpy()
govt  = macro["realgovt"].to_numpy()
pop   = macro["pop"].to_numpy()         # millions
unemp = macro["unemp"].to_numpy()       # %

print("NumPy", np.__version__)
print(type(gdp), gdp.shape, gdp.dtype)



NumPy 2.5.3
<class 'numpy.ndarray'> (203,) float64


## 1. Why arrays?

A list is a general container: it can hold anything, so Python cannot assume it holds numbers. An array holds **one type** of value in a **fixed shape**, which is what makes element-wise arithmetic possible — and fast.


In [15]:
prices_list  = [100, 102, 105]
prices_array = np.array([100, 102, 105])

print("list  * 2 :", prices_list * 2)      # repeats the list!
print("array * 2 :", prices_array * 2)     # multiplies each element


list  * 2 : [100, 102, 105, 100, 102, 105]
array * 2 : [200 204 210]


Same symbol, completely different meaning. For a list, `* 2` means *repeat*; for an array it means *multiply every element*. Mixing the two up is the first NumPy trap.

**Speed.** Below, the same growth-rate calculation on one million simulated values, once with a list comprehension (the Session 1 way) and once vectorised.


In [16]:
levels = np.random.default_rng(0).uniform(100, 200, size=1_000_000)
as_list = levels.tolist()

%timeit [(as_list[i] / as_list[i-1] - 1) for i in range(1, len(as_list))]
%timeit levels[1:] / levels[:-1] - 1



54.8 ms ± 4.3 ms per loop (mean ± std. dev. of 7 runs, 10 loops each)
355 μs ± 41.7 μs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)


The vectorised version is typically tens of times faster: the loop runs inside NumPy's compiled code instead of in Python. On a 203-row dataset you will not notice; on a household survey or a simulation you will.


## 2. Creating and inspecting arrays

Four attributes tell you what you are holding: `shape` (the size along each axis), `ndim` (how many axes), `size` (how many values) and `dtype` (what kind of values).


In [17]:
a = np.arange(2000, 2010)           # like range(), but returns an array
b = np.linspace(0, 1, 5)            # 5 evenly spaced values from 0 to 1, both ends included
z = np.zeros((3, 4))                # 3 rows, 4 columns

for name, arr in [("a", a), ("b", b), ("z", z)]:
    print(f"{name}: shape={arr.shape}, ndim={arr.ndim}, size={arr.size}, dtype={arr.dtype}")



a: shape=(10,), ndim=1, size=10, dtype=int64
b: shape=(5,), ndim=1, size=5, dtype=float64
z: shape=(3, 4), ndim=2, size=12, dtype=float64


**Trap: the dtype is fixed.** An integer array cannot hold a decimal — NumPy silently cuts it off.


In [18]:
counts = np.array([1, 2, 3])        # dtype int64
counts[0] = 2.9
counts                              # 2.9 became 2, with no warning


array([2, 2, 3])

**Trap: how NumPy 2 displays single numbers.** When a cell ends with a single number from NumPy, Jupyter shows it with its type attached:


In [19]:
gdp.mean()


np.float64(7221.171901477834)

In [20]:
print(gdp.mean())                  # print() shows just the number
print(f"{gdp.mean():,.1f}")        # and an f-string formats it for reporting
print(np.unique(year[:9]).tolist())  # .tolist() turns array values into plain Python numbers


7221.171901477834
7,221.2
[1959, 1960, 1961]


`np.float64(...)` is not an error — it is NumPy 2 telling you the number's type. For reports, use `print`, an f-string, or `.tolist()`.


## 3. Indexing, slicing and boolean masks

Indexing follows the list rules from Session 1: positions start at 0, a slice `a:b` stops *before* `b`, negative positions count from the end. Slices also take a **step**: `a:b:step`.


In [21]:
print("first quarter :", year[0], qtr[0], gdp[0])
print("last quarter  :", year[-1], qtr[-1], gdp[-1])
print("first 4 values:", gdp[:4])

q1_gdp = gdp[::4]                   # every 4th value, starting at position 0 -> every Q1
print("Q1 observations:", q1_gdp.size, "| first years:", year[::4][:5])


first quarter : 1959 1 2710.349
last quarter  : 2009 3 12990.341
first 4 values: [2710.349 2778.801 2775.488 2785.204]
Q1 observations: 51 | first years: [1959 1960 1961 1962 1963]


**Trap: a slice is a view, not a copy.** Changing a slice changes the original array. (Lists copy when sliced; arrays do not.)


In [22]:
demo = np.array([10, 20, 30, 40])
first_two = demo[:2]
first_two[0] = 99
print("demo after changing the slice:", demo)

safe = demo[:2].copy()              # .copy() when you want an independent array
safe[0] = -1
print("demo after changing the copy :", demo)


demo after changing the slice: [99 20 30 40]
demo after changing the copy : [99 20 30 40]


**Boolean masks.** A comparison on an array gives an array of `True`/`False` — a *mask*. Using the mask inside `[...]` keeps only the `True` positions. Because `True` counts as 1, `.sum()` counts matches and `.mean()` gives the share.


In [23]:
high = unemp > 9
print(high[:8])
print("quarters with unemployment above 9%:", high.sum())
print("their GDP levels:", gdp[high].round(0))


[False False False False False False False False]
quarters with unemployment above 9%: 8
their GDP levels: [ 5889.  5866.  5871.  5944.  6078.  6197. 12902. 12990.]


In [24]:
# Combine conditions with & (and) and | (or) - parentheses are required
eighties_high = (year >= 1980) & (year < 1990) & (unemp > 9)
print(eighties_high.sum(), "quarters in the 1980s above 9%")


6 quarters in the 1980s above 9%


### ✏️ Exercise — 1.1 High-unemployment quarters

How many quarters had unemployment **above 8%**? What **share** of the sample is that (in %)? In which **years** did it happen? *Hints:* `.sum()`, `.mean()`, and `np.unique(year[mask]).tolist()`. (Check: 16 quarters, in five different years.)

In [39]:
 # your code here
mask = unemp > 8
print("liczba kwartałów:", mask.sum())                 # oczekiwane: 16
print(f"udział: {mask.mean() * 100:.1f}%")
print("lata:", np.unique(year[mask]).tolist())          # oczekiwane: 5 lat


liczba kwartałów: 16
udział: 7.9%
lata: [1975, 1981, 1982, 1983, 2009]


## 4. Vectorised arithmetic on economic series

Here is Session 1's growth-rate loop in one line. The trick: `gdp[1:]` is every quarter *except the first*, `gdp[:-1]` every quarter *except the last* — dividing them lines up each quarter with the one before.


In [46]:
growth_q = (gdp[1:] / gdp[:-1] - 1) * 100       # quarterly growth, %
print("length:", gdp.size, "->", growth_q.size)   # one fewer: the first quarter has no predecessor
print(growth_q[:4].round(3))


length: 203 -> 202
[ 2.526 -0.119  0.35   2.244]


Economists often use **log differences** instead: for small changes, $\ln(x_t) - \ln(x_{t-1}) \approx x_t/x_{t-1} - 1$. NumPy applies `np.log` to every element, and `np.diff` takes consecutive differences.


In [47]:
log_growth = np.diff(np.log(gdp)) * 100
print("largest gap between the two measures:", np.abs(log_growth - growth_q).max().round(3), "percentage points")


largest gap between the two measures: 0.075 percentage points


In [51]:
growth_q
worst_q = growth_q.argmin()
print("worst quarter:", year[worst_q+1],qtr[worst_q+1],"with growth", growth_q[worst_q].round(3), "%")


worst quarter: 1980 2 with growth -2.049 %


In [52]:
worst5 = growth_q.argsort()[:5]          # pozycje 5 najgorszych kwartałów

pd.DataFrame({
    "year": year[worst5 + 1],
    "quarter": qtr[worst5 + 1],
    "growth_%": growth_q[worst5].round(3),
})

,year,quarter,growth_%
0,1980,2,-2.049
1,2009,1,-1.647
2,1982,1,-1.641
3,2008,4,-1.371
4,1960,4,-1.282


**Summary statistics** are methods of the array; `argmin`/`argmax` return the *position* of the extreme value, which you can use to look up the date.


In [53]:
print(f"mean {growth_q.mean():.3f}%, std {growth_q.std():.3f}%, min {growth_q.min():.3f}%, max {growth_q.max():.3f}%")

worst = growth_q.argmin()                       # position in growth_q ...
print(f"worst quarter: {year[worst + 1]}Q{qtr[worst + 1]}")   # ... is position +1 in the original series


mean 0.783%, std 0.884%, min -2.049%, max 3.934%
worst quarter: 1980Q2


Note the `+ 1`: `growth_q[0]` is the growth *into* the second quarter. Keeping track of this offset is the price of the one-line formula.

**Missing values.** `np.nan` ("not a number") marks a missing value, and it spreads: any calculation involving it returns `nan`. The `nan`-functions skip missing values instead.


In [44]:
x = np.array([2.0, np.nan, 4.0])
print("mean   :", x.mean())
print("nanmean:", np.nanmean(x))


mean   : nan
nanmean: 3.0


### ✏️ Exercise — 2.1 Annualised growth and recession quarters

Quarterly growth rates are usually reported **annualised**: $\left(\frac{GDP_t}{GDP_{t-1}}\right)^4 - 1$. Compute the annualised growth series (in %) as `growth_ann`, report its mean and standard deviation, and count the quarters with **negative** growth (and their share of all quarters). (Check: 28 negative quarters.)

In [45]:
 # your code here
growth_ann = ((gdp[1:] / gdp[:-1]) ** 4 - 1) * 100
print(f"średnia {growth_ann.mean():.2f}%, odch. std. {growth_ann.std():.2f}%")

neg = growth_ann < 0
print("kwartały z ujemnym wzrostem:", neg.sum())        # oczekiwane: 28
print(f"udział: {neg.mean() * 100:.1f}%")


średnia 3.22%, odch. std. 3.61%
kwartały z ujemnym wzrostem: 28
udział: 13.9%


### ✏️ Exercise — 2.2 Per-capita GDP

`gdp` is in billions and `pop` in millions, so `gdp / pop` is real GDP per person in **thousands** of 2005 dollars. Compute it, print the first and last values, and the **average annual growth** of per-capita GDP over the whole sample (the sample spans `(gdp.size - 1) / 4` years). Compare it with the average annual growth of total GDP. In one markdown sentence: what explains the gap? (Check: per-capita growth is about 2.0% a year.)

In [54]:
  # your code here
# PKB per capita (tysiące dolarów z 2005 r.)
gdp_pc = gdp / pop

# pierwsza i ostatnia wartość
print(f"pierwsza: {gdp_pc[0]:.2f}, ostatnia: {gdp_pc[-1]:.2f}")

# średni roczny wzrost przez całą próbę (próba obejmuje (size-1)/4 lat)
lata = (gdp_pc.size - 1) / 4
wzrost_pc  = (gdp_pc[-1] / gdp_pc[0]) ** (1 / lata) - 1
wzrost_gdp = (gdp[-1] / gdp[0]) ** (1 / lata) - 1

print(f"średni roczny wzrost per capita: {wzrost_pc:.2%}")
print(f"średni roczny wzrost całkowitego PKB: {wzrost_gdp:.2%}")



pierwsza: 15.30, ostatnia: 42.17
średni roczny wzrost per capita: 2.03%
średni roczny wzrost całkowitego PKB: 3.15%


## 5. Two dimensions and broadcasting

Stack several series side by side and you get a **2-D array**: rows are quarters, columns are variables. `axis=0` means "down the rows" (one result per column); `axis=1` means "across the columns" (one result per row).


In [55]:
X = np.column_stack([cons, inv, govt])          # columns: consumption, investment, government
print("shape:", X.shape)
print("column means (axis=0):", X.mean(axis=0).round(0))
print("first 3 row sums (axis=1):", X.sum(axis=1)[:3].round(0))


shape: (203, 3)
column means (axis=0): [4825. 1013.  663.]
first 3 row sums (axis=1): [2464. 2526. 2532.]


**Broadcasting** is how NumPy combines arrays of different shapes. Shapes are compared from the **right**: two sizes fit if they are equal or one of them is 1. A shape `(3,)` array therefore fits against `(203, 3)`: it is stretched down all 203 rows.


In [56]:
indexed = X / X[0] * 100        # X[0] has shape (3,): each column divided by its own first value
print(indexed[[0, -1]].round(1))   # first and last rows: everything starts at 100


[[100.  100.  100. ]
 [542.1 518.1 222.1]]


Now divide each component by **that quarter's GDP**. `gdp` has shape `(203,)` — and that fails:


In [34]:
try:
    X / gdp
except ValueError as e:
    print("ValueError:", e)


ValueError: operands could not be broadcast together with shapes (203,3) (203,) 


Compared from the right, `3` (columns of `X`) meets `203` (length of `gdp`): no fit. The fix is to turn `gdp` into a **column** of shape `(203, 1)` with `gdp[:, None]`; then the `1` stretches across the 3 columns.


In [35]:
print(gdp.shape, "->", gdp[:, None].shape)
shares = X / gdp[:, None]
print(shares.shape)


(203,) -> (203, 1)
(203, 3)


Rule of thumb: dividing by **one value per column** works directly; dividing by **one value per row** needs `[:, None]`.


In [57]:
# Składanie kilku serii obok siebie w tablicę 2D (kolumny: konsumpcja, inwestycje, wydatki rządu)
X = np.column_stack((cons, inv, govt))

print("shape:", X.shape)
print("column means (axis=0):", X.mean(axis=0).round(1))
print("first 3 row sums (axis=1):", X.sum(axis=1)[:3].round(0))

shape: (203, 3)
column means (axis=0): [4825.3 1012.9  663.3]
first 3 row sums (axis=1): [2464. 2526. 2532.]


In [58]:
gdp_pc = gdp / pop  # gdp jest w miliardach, a pop w milionach, więc gdp / pop daje realne PKB per capita


### ✏️ Exercise — 3.1 Shares of GDP

Using `shares`, compute the **average share** of consumption, investment and government in GDP over the sample (in %, one decimal), and the **sum** of the three average shares. The sum is not 100%. In a markdown cell, give two reasons why. *Hint:* which component of $Y = C + I + G + NX$ is missing, and are chained-dollar real series additive?

In [63]:
 # your code here
avg_shares = (shares.mean(axis=0) * 100).round(1)

print("konsumpcja, inwestycje, rząd:", avg_shares)
print("suma:", avg_shares.sum().round(1))


konsumpcja, inwestycje, rząd: [65.8 13.3 10.5]
suma: 89.6


In [64]:
print(shares)

[[0.62995577 0.10585279 0.17342601]
 [0.62390218 0.11186803 0.17320456]
 [0.63116828 0.10420726 0.1769995 ]
 [0.62964867 0.10748082 0.17379409]
 [0.62173004 0.11648773 0.16230613]
 [0.63255233 0.10519089 0.16243354]
 [0.62901943 0.10439334 0.16719701]
 [0.63804674 0.09268626 0.16999618]
 [0.63410167 0.09449452 0.16878661]
 [0.63171896 0.09966765 0.16724483]
 [0.62468754 0.10629968 0.1692108 ]
 [0.62448159 0.10593721 0.16875409]
 [0.62001009 0.1102753  0.1718636 ]
 [0.62077672 0.10801645 0.17067395]
 [0.62006817 0.10894177 0.17420944]
 [0.62733768 0.10502931 0.17284345]
 [0.62341476 0.10942741 0.16647645]
 [0.62157929 0.10964811 0.16290415]
 [0.61839342 0.11113121 0.16877684]
 [0.61887302 0.11165013 0.16305923]
 [0.61724031 0.11368934 0.15867195]
 [0.62095246 0.11188161 0.15583043]
 [0.62387709 0.11300438 0.15252381]
 [0.62390059 0.11361156 0.14994448]
 [0.62248116 0.12204618 0.14447376]
 [0.62093851 0.12040511 0.1427993 ]
 [0.61903839 0.12222474 0.1456008 ]
 [0.62145309 0.11989563 0.14

Suma nie wynosi 100%, ponieważ: (1) brakuje eksportu netto (NX) w tożsamości Y = C + I + G + NX, a USA mają deficyt handlowy, więc NX jest ujemny i C + I + G przekracza 100% PKB; (2) szeregi realne w cenach łańcuchowych (chained dollars) nie są addytywne, więc suma składowych nie musi równać się PKB.

The sum is not 100% because (1) net exports (NX) are missing from Y = C + I + G + NX, and the US runs a trade deficit, so NX < 0 and C + I + G exceeds 100% of GDP; (2) chained-dollar real series are not additive, so components do not sum to the total.

## 6. Stretch: simulating uncertainty

Vectorisation pays off most in simulation. Below: 1,000 possible paths of GDP over the next 20 quarters, drawing each quarter's growth at random from a normal distribution with the historical mean and standard deviation. `default_rng(42)` fixes the random seed, so everyone gets the same numbers.


In [66]:
rng = np.random.default_rng(42)
mu, sd = growth_q.mean() / 100, growth_q.std() / 100

draws = rng.normal(mu, sd, size=(1000, 20))              # 1000 paths x 20 quarters, in one call
paths = gdp[-1] * np.cumprod(1 + draws, axis=1)          # compound growth along each row
print(paths.shape)


(1000, 20)


### ✏️ Exercise — 3.2 (stretch) How uncertain is GDP five years ahead?

From `paths`, take the final quarter of every path and compute its 5th, 50th and 95th **percentiles** (`np.percentile`). Then, in one markdown sentence: name one assumption of this simulation that you find unrealistic for an economy.

In [67]:
# your code here
final = paths[:, -1]                      # PKB w ostatnim kwartale, po jednej wartości na ścieżkę (1000 liczb)
p5, p50, p95 = np.percentile(final, [5, 50, 95])

print(f"5. percentyl:  {p5:.1f}")
print(f"50. percentyl: {p50:.1f}")
print(f"95. percentyl: {p95:.1f}")




5. percentyl:  14219.9
50. percentyl: 15185.1
95. percentyl: 16223.1


Symulacja zakłada, że wzrost w każdym kwartale jest losowany niezależnie od poprzednich kwartałów. W prawdziwej gospodarce tak nie jest: jeśli gospodarka rośnie szybko lub wpada w recesję, to zwykle trwa to kilka kwartałów z rzędu, a nie jest przypadkowe z kwartału na kwartał.

One unrealistic assumption is that quarterly growth rates are independent draws from the same normal distribution, ignoring that real growth is autocorrelated, that recessions and booms cluster, and that the mean and volatility change over time.


## 7. Before Session 3 (Wed 14 Oct, 10:15–12:40, **remote on MS Teams**)

1. **Self-check quiz** for this session on eNauczanie (ungraded, ~10 minutes): lists vs arrays, the length of a growth series, views, broadcasting, missing values.
2. **Check that MS Teams works** with your PG account — Session 3 is our first remote session; the link is in Lab Session 2.
3. **Read the Session 3 briefing notebook** (posted by Mon 12 Oct, ~20 minutes) and download `lfs_sample_2023.csv`. Session 3 includes **Portfolio task 1**, your first graded task (due Tue 20 Oct).
4. Have not posted your introduction in the forum yet? It is still open.
